# Un VLM comme juge de légendes, et démo

*Projet 7 : VLMs légers pour légender ou juger des images.*

Notation de légendes par les VLMs (1 à 5), comparaison avec nos propres notes selon le niveau de quantification, puis démo Gradio.

GPU requis dans Colab : *Exécution → Modifier le type d'exécution → T4 GPU*. Exécute les cellules dans l'ordre ; ce qui a déjà été calculé est repris depuis le Drive partagé.

## Mise en place (commune aux trois notebooks)

> **Dossier partagé** : Wafae crée `projet7_vlm` dans son Google Drive et le partage avec Meriem et Fatima (en éditrices). Meriem et Fatima l'ajoutent à *Mon Drive* : clic droit → *Organiser* → *Ajouter un raccourci à Drive*. Ainsi `MyDrive/projet7_vlm` pointe vers le même dossier pour les trois.
>
> **Ne pas modifier** `SEED`, `N_IMAGES`, `MAX_SIDE` sans accord de l'équipe : sinon les images ne sont plus les mêmes.

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate bitsandbytes datasets pycocoevalcap gradio num2words scipy

In [ ]:
import os, gc, json, re, time, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), "Aucun GPU détecté : active un GPU (Exécution → Modifier le type d'exécution)."
print("GPU :", torch.cuda.get_device_name(0))
print("Mémoire GPU totale : %.1f Go" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# bfloat16 seulement sur GPU Ampere ou plus récent (A100, L4...). Sur T4 → float16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Type des poids en 16 bits :", DTYPE)

# ------------------------- CONFIGURATION -------------------------
SEED = 42
N_IMAGES = 300            # taille du sous-ensemble évalué (le sujet conseille 500 à 2000 ; 300 ≈ 1 h sur T4)
MAX_SIDE = 512            # redimensionnement des images (côté le plus long, en pixels)
MAX_NEW_TOKENS = 40       # longueur max d'une légende

MODELS = {
    "Qwen2-VL-2B": "Qwen/Qwen2-VL-2B-Instruct",
    "SmolVLM-2B":  "HuggingFaceTB/SmolVLM-Instruct",
}
PRECISIONS = ["16bit", "8bit", "4bit"]

CAPTION_PROMPT = "Describe this image in one short sentence."

# Option : sauvegarder sur Google Drive pour ne rien perdre en cas de déconnexion
USE_DRIVE = True   # dossier partagé de l'équipe : MyDrive/projet7_vlm
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet7_vlm")
else:
    ROOT = Path("/content/projet7_vlm") if Path("/content").exists() else Path("projet7_vlm")

DATA_DIR = ROOT / "data"
IMG_DIR = DATA_DIR / "images"
RES_DIR = ROOT / "results"
for d in (IMG_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Dossier de travail :", ROOT)

## 2. Préparation des données — Flickr8k

Flickr8k contient ~8 000 images, chacune avec **5 légendes humaines**. On prend un sous-ensemble aléatoire (graine fixe) de la partie *test*, on **redimensionne** les images (côté max = `MAX_SIDE`) et on les enregistre sur disque : **les mêmes images** serviront à toutes les configurations (16 / 8 / 4 bits), ce qui rend la comparaison équitable.

In [ ]:
from datasets import load_dataset

def prepare_image(img, max_side=MAX_SIDE):
    # Convertit en RGB et réduit l'image pour que son plus grand côté fasse max_side pixels.
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side), Image.BICUBIC)
    return img

META_PATH = DATA_DIR / "subset.json"

if META_PATH.exists():
    subset = json.loads(META_PATH.read_text())
    print(f"Sous-ensemble déjà préparé : {len(subset)} images")
else:
    ds = load_dataset("jxie/flickr8k", split="test")          # 1000 images, colonnes image + caption_0..4
    caption_cols = sorted(c for c in ds.column_names if c.startswith("caption"))
    idx = sorted(random.Random(SEED).sample(range(len(ds)), min(N_IMAGES, len(ds))))
    subset = []
    for i in tqdm(idx, desc="Préparation des images"):
        ex = ds[i]
        img_id = f"img_{i:04d}"
        prepare_image(ex["image"]).save(IMG_DIR / f"{img_id}.jpg", quality=95)
        subset.append({"id": img_id, "refs": [ex[c].strip() for c in caption_cols]})
    META_PATH.write_text(json.dumps(subset, indent=1, ensure_ascii=False))
    print(f"{len(subset)} images préparées dans {IMG_DIR}")

REFS = {s["id"]: s["refs"] for s in subset}
IDS = [s["id"] for s in subset]

def load_img(img_id):
    return Image.open(IMG_DIR / f"{img_id}.jpg").convert("RGB")

### Chargement des VLMs

In [ ]:
import transformers
from packaging import version
from transformers import AutoProcessor, BitsAndBytesConfig
# Nouvelles versions de transformers : argument `dtype` ; anciennes : `torch_dtype`
DTYPE_KW = {"dtype": DTYPE} if version.parse(transformers.__version__) >= version.parse("4.56") else {"torch_dtype": DTYPE}
print("transformers", transformers.__version__)
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

def quant_config(precision):
    if precision == "16bit":
        return None
    if precision == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "4bit":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    raise ValueError(precision)

def load_vlm(name, precision):
    model_id = MODELS[name]
    # On limite la résolution vue par le modèle pour garder un nombre de tokens image raisonnable
    if "Qwen" in model_id:
        proc_kwargs = dict(min_pixels=128 * 28 * 28, max_pixels=512 * 28 * 28)
    elif "SmolVLM" in model_id:
        proc_kwargs = dict(size={"longest_edge": 2 * 384})
    else:
        proc_kwargs = {}
    try:
        processor = AutoProcessor.from_pretrained(model_id, **proc_kwargs)
    except Exception:
        processor = AutoProcessor.from_pretrained(model_id)
    model = AutoVLM.from_pretrained(model_id, device_map="cuda:0",
                                    quantization_config=quant_config(precision), **DTYPE_KW)
    model.eval()
    return model, processor

def unload(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

@torch.inference_mode()
def vlm_generate(model, processor, image, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=[image], return_tensors="pt")
    inputs = {k: (v.to(model.device, DTYPE) if torch.is_floating_point(v) else v.to(model.device))
              for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

def clean_caption(text):
    # Garde la première ligne / phrase et retire un éventuel préfixe « Assistant: ».
    text = re.sub(r"^\s*(assistant\s*:)\s*", "", text.strip(), flags=re.I)
    text = text.split("\n")[0].strip()
    m = re.match(r"(.{15,}?[.!?])(\s|$)", text)
    return (m.group(1) if m else text).strip()

### CLIPScore (point de comparaison)

In [ ]:
# CLIPScore (Hessel et al., 2021) : 2.5 * max(cos(image, texte), 0)
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").half().cuda().eval()

def _as_tensor(e):
    # Selon la version de transformers, get_*_features renvoie un tenseur ou un objet de sortie
    if torch.is_tensor(e):
        return e
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(e, attr, None) is not None:
            return getattr(e, attr)
    raise TypeError(type(e))
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.inference_mode()
def clip_image_emb(ids, bs=32):
    embs = []
    for i in range(0, len(ids), bs):
        imgs = [load_img(x) for x in ids[i:i + bs]]
        px = clip_proc(images=imgs, return_tensors="pt")["pixel_values"].cuda().half()
        e = clip_model.get_image_features(pixel_values=px)
        embs.append(torch.nn.functional.normalize(_as_tensor(e).float(), dim=-1).cpu())
    return dict(zip(ids, torch.cat(embs)))

@torch.inference_mode()
def clip_text_emb(texts, bs=64):
    embs = []
    for i in range(0, len(texts), bs):
        t = clip_proc(text=list(texts[i:i + bs]), return_tensors="pt", padding=True,
                      truncation=True, max_length=77)
        e = clip_model.get_text_features(input_ids=t["input_ids"].cuda(), attention_mask=t["attention_mask"].cuda())
        embs.append(torch.nn.functional.normalize(_as_tensor(e).float(), dim=-1).cpu())
    return torch.cat(embs)

IMG_EMB = clip_image_emb(IDS)

def clip_scores(ids, captions):
    T = clip_text_emb([str(c) for c in captions])
    I = torch.stack([IMG_EMB[i] for i in ids])
    return (2.5 * (I * T).sum(-1).clamp(min=0)).numpy()

## 7. Bonus option B — Le VLM comme juge

On demande au VLM de **noter de 1 à 5** si une légende décrit bien l'image. Pour chaque image du jeu de jugement, on lui soumet trois légendes :
- `humaine` : une légende Flickr8k (devrait être bien notée) ;
- `vlm` : une légende générée en partie 4 (qualité variable) ;
- `mauvaise` : la légende d'une **autre** image (devrait être mal notée).

On compare ensuite ses notes à un **jugement humain** avec les corrélations de **Spearman**, **Pearson** et **Kendall**, pour chaque niveau de quantification.

**Annotation humaine** : l'outil ci-dessous affiche chaque paire image/légende et tu cliques une note de 1 à 5 (≈ 10 min pour 120 paires). Si tu n'annotes pas, le notebook utilise un **proxy** (humaine = 5, mauvaise = 1, paires `vlm` exclues) — à signaler clairement dans le rapport.

In [ ]:
# Légendes VLM pour les paires « vlm » : on prend celles de Wafae (Qwen2-VL-2B, 16 bits) si elles existent,
# sinon on les génère ici pour les N_JUDGE premières images (≈ 1 min).
N_JUDGE = 40
VLM_CAPTION_CONFIG = ("Qwen2-VL-2B", "16bit")
cap_path = RES_DIR / f"captions_{VLM_CAPTION_CONFIG[0]}__{VLM_CAPTION_CONFIG[1]}.csv"
own_path = RES_DIR / "judge_vlm_captions.csv"
if own_path.exists():
    VLM_CAPS = pd.read_csv(own_path).set_index("id")["caption"].to_dict()
elif cap_path.exists():
    VLM_CAPS = pd.read_csv(cap_path).set_index("id")["caption"].to_dict()
else:
    m, p = load_vlm(*VLM_CAPTION_CONFIG)
    VLM_CAPS = {i: clean_caption(vlm_generate(m, p, load_img(i), CAPTION_PROMPT)) for i in tqdm(IDS[:N_JUDGE])}
    unload(m, p); del m, p; gc.collect(); torch.cuda.empty_cache()
# On fige ces légendes pour que les paires restent identiques d'une exécution à l'autre
pd.DataFrame({"id": IDS[:N_JUDGE], "caption": [VLM_CAPS[i] for i in IDS[:N_JUDGE]]}).to_csv(own_path, index=False)
VLM_CAPS = pd.read_csv(own_path).set_index("id")["caption"].to_dict()
print(len(VLM_CAPS), "légendes VLM prêtes")

In [ ]:
N_JUDGE = 40                                   # images → 3 × N_JUDGE paires
rng = random.Random(SEED)
judge_ids = IDS[:N_JUDGE]
pairs = []
for k, img_id in enumerate(judge_ids):
    other = judge_ids[(k + rng.randint(1, N_JUDGE - 1)) % N_JUDGE]
    pairs += [{"image_id": img_id, "type": "humaine", "caption": REFS[img_id][1]},
              {"image_id": img_id, "type": "vlm", "caption": VLM_CAPS[img_id]},
              {"image_id": img_id, "type": "mauvaise", "caption": REFS[other][1]}]
rng.shuffle(pairs)
pairs = pd.DataFrame(pairs)
pairs.insert(0, "pair_id", range(len(pairs)))

ANNOTATRICE = "fatima"   # chaque membre qui annote met SON prénom ici (wafae, meriem, fatima)
HUMAN_PATH = RES_DIR / f"notes_humaines_{ANNOTATRICE}.csv"
if HUMAN_PATH.exists():
    human = pd.read_csv(HUMAN_PATH)
else:
    human = pairs.assign(human_score=np.nan)
    human.to_csv(HUMAN_PATH, index=False)
print(f"{len(pairs)} paires — déjà annotées : {human['human_score'].notna().sum()}")

In [ ]:
# Outil d'annotation (clique une note ; la sauvegarde est automatique)
import io
import ipywidgets as widgets
from IPython.display import display

state = {"i": int(human["human_score"].notna().sum())}
img_w = widgets.Image(format="jpeg", width=380)
cap_w = widgets.HTML()
prog_w = widgets.HTML()
buttons = [widgets.Button(description=str(s), layout=widgets.Layout(width="60px")) for s in range(1, 6)]

def show():
    i = state["i"]
    if i >= len(human):
        prog_w.value = "<b>✅ Annotation terminée.</b>"; cap_w.value = ""; return
    r = human.iloc[i]
    buf = io.BytesIO(); load_img(r.image_id).save(buf, format="JPEG"); img_w.value = buf.getvalue()
    cap_w.value = f"<p style='font-size:16px'><b>Légende :</b> {r.caption}</p><p>1 = complètement faux … 5 = parfait</p>"
    prog_w.value = f"Paire {i + 1} / {len(human)}"

def on_click(b):
    if state["i"] < len(human):
        human.loc[state["i"], "human_score"] = int(b.description)
        human.to_csv(HUMAN_PATH, index=False)
        state["i"] += 1
    show()

for b in buttons:
    b.on_click(on_click)
show()
display(widgets.VBox([prog_w, img_w, cap_w, widgets.HBox(buttons)]))

### Test rapide du prompt

In [ ]:
JUDGE_PROMPT = ('Caption: "{caption}"\n'
                "On a scale from 1 to 5, how well does this caption describe the image? "
                "1 = completely wrong, 5 = perfect. Answer with a single digit.")

def parse_score(text):
    m = re.search(r"[1-5]", text)
    return int(m.group()) if m else np.nan


# Test du prompt sur 6 paires avec un seul modèle en 4 bits (rapide)
m, p = load_vlm("Qwen2-VL-2B", "4bit")
for r in pairs.head(6).itertuples():
    out = vlm_generate(m, p, load_img(r.image_id), JUDGE_PROMPT.format(caption=r.caption), max_new_tokens=5)
    print(f"[{r.type:8s}] note = {parse_score(out)}  (réponse brute : {out!r})  — {r.caption}")
unload(m, p); del m, p; gc.collect(); torch.cuda.empty_cache()

## Notes du juge pour les six configurations

In [ ]:
JUDGE_PROMPT = ('Caption: "{caption}"\n'
                "On a scale from 1 to 5, how well does this caption describe the image? "
                "1 = completely wrong, 5 = perfect. Answer with a single digit.")

def parse_score(text):
    m = re.search(r"[1-5]", text)
    return int(m.group()) if m else np.nan

JUDGE_MODELS = list(MODELS)     # les deux modèles, en 16 / 8 / 4 bits
for name in JUDGE_MODELS:
    for prec in PRECISIONS:
        path = RES_DIR / f"judge_{name}__{prec}.csv"
        if path.exists():
            print(f"[déjà calculé] juge {name} {prec}"); continue
        model, processor = load_vlm(name, prec)
        scores = []
        for r in tqdm(pairs.itertuples(), total=len(pairs), desc=f"juge {name} {prec}"):
            out = vlm_generate(model, processor, load_img(r.image_id),
                               JUDGE_PROMPT.format(caption=r.caption), max_new_tokens=5)
            scores.append({"pair_id": r.pair_id, "raw": out, "vlm_score": parse_score(out)})
        pd.DataFrame(scores).to_csv(path, index=False)
        unload(model, processor); del model, processor
        gc.collect(); torch.cuda.empty_cache()

## Corrélation avec nos notes

In [ ]:
from scipy.stats import spearmanr, pearsonr, kendalltau

# Moyenne des notes de toutes les annotatrices (fichiers notes_humaines_<prénom>.csv du dossier partagé)
_files = sorted(RES_DIR.glob("notes_humaines_*.csv"))
_all = pd.concat([pd.read_csv(f)[["pair_id", "human_score"]].assign(annotatrice=f.stem.split("_")[-1]) for f in _files])
human = pd.read_csv(HUMAN_PATH)
_mean = _all.dropna(subset=["human_score"]).groupby("pair_id")["human_score"].mean().reset_index()
print("Annotatrices :", [f.stem.split("_")[-1] for f in _files])

# Accord entre annotatrices (si au moins deux)
_wide = _all.pivot_table(index="pair_id", columns="annotatrice", values="human_score")
_names = list(_wide.columns)
for a in range(len(_names)):
    for b in range(a + 1, len(_names)):
        _d = _wide[[_names[a], _names[b]]].dropna()
        if len(_d) >= 10:
            print(f"Accord {_names[a]} / {_names[b]} : Spearman = {spearmanr(_d.iloc[:, 0], _d.iloc[:, 1])[0]:.2f} sur {len(_d)} paires")

if len(_mean) >= 20:
    gt = _mean; source = f"moyenne des notes humaines ({len(_files)} annotatrice(s))"
else:
    proxy = human[human["type"] != "vlm"].copy()
    proxy["human_score"] = proxy["type"].map({"humaine": 5, "mauvaise": 1})
    gt = proxy[["pair_id", "human_score"]]; source = "PROXY (humaine=5 / mauvaise=1) — annote pour un vrai résultat"
print("Vérité terrain utilisée :", source, f"({len(gt)} paires)")

pairs["CLIPScore"] = clip_scores(pairs["image_id"].tolist(), pairs["caption"].tolist())
corr_rows = []
for name in JUDGE_MODELS:
    for prec in PRECISIONS:
        j = pd.read_csv(RES_DIR / f"judge_{name}__{prec}.csv")
        d = pairs.merge(j, on="pair_id").merge(gt, on="pair_id").dropna(subset=["vlm_score"])
        corr_rows.append({"juge": f"{name} {prec}", "n": len(d),
                          "réponses valides %": 100 * j["vlm_score"].notna().mean(),
                          "Spearman": spearmanr(d.vlm_score, d.human_score)[0],
                          "Pearson": pearsonr(d.vlm_score, d.human_score)[0],
                          "Kendall": kendalltau(d.vlm_score, d.human_score)[0],
                          "note moy. humaine": d[d.type == "humaine"].vlm_score.mean(),
                          "note moy. vlm": d[d.type == "vlm"].vlm_score.mean(),
                          "note moy. mauvaise": d[d.type == "mauvaise"].vlm_score.mean()})
d = pairs.merge(gt, on="pair_id")
corr_rows.append({"juge": "CLIPScore (référence)", "n": len(d),
                  "Spearman": spearmanr(d.CLIPScore, d.human_score)[0],
                  "Pearson": pearsonr(d.CLIPScore, d.human_score)[0],
                  "Kendall": kendalltau(d.CLIPScore, d.human_score)[0]})
corr = pd.DataFrame(corr_rows)
corr.to_csv(RES_DIR / "correlations_juge.csv", index=False)
corr.round(3)

In [ ]:
colors = {"16bit": "#2a6fdb", "8bit": "#f2a541", "4bit": "#3bb273"}
c = corr[corr["juge"] != "CLIPScore (référence)"]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(c["juge"], c["Spearman"], color=[colors[j.split()[-1]] for j in c["juge"]])
ax.bar_label(bars, fmt="%.2f")
clip_s = corr[corr["juge"] == "CLIPScore (référence)"]["Spearman"].values[0]
ax.axhline(clip_s, ls="--", color="gray", label=f"CLIPScore ({clip_s:.2f})")
ax.set_ylabel("Corrélation de Spearman avec l'humain"); ax.set_ylim(0, 1)
ax.set_title("Accord VLM-juge / humain selon la quantification")
plt.xticks(rotation=20, ha="right"); ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(RES_DIR / "correlation_juge.png", dpi=150); plt.show()

### À analyser
- Le VLM note-t-il comme nous ? Quel modèle est le meilleur juge ?
- La quantification change-t-elle l'accord ?
- Le VLM fait-il mieux que le CLIPScore ?

## 8. Démo finale — Gradio

Charge une image, choisis le modèle, la **précision (16 / 8 / 4 bits)** et le mode :
- **Légende** : le VLM décrit l'image ;
- **Score** : le VLM note de 1 à 5 une légende (la tienne, ou à défaut celle qu'il génère).

Un seul modèle est gardé en mémoire à la fois. Le lien public `gradio.live` s'affiche sous la cellule.

In [ ]:
import gradio as gr

for _v in ("clip_model",):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

_cur = {"key": None, "model": None, "proc": None}

def get_model(name, prec):
    if _cur["key"] != (name, prec):
        if _cur["model"] is not None:
            _cur["model"] = _cur["proc"] = None
            gc.collect(); torch.cuda.empty_cache()
        _cur["model"], _cur["proc"] = load_vlm(name, prec)
        _cur["key"] = (name, prec)
    return _cur["model"], _cur["proc"]

def demo_fn(image, name, prec, mode, user_caption):
    if image is None:
        return "Charge une image.", ""
    image = prepare_image(image)
    model, proc = get_model(name, prec)
    torch.cuda.synchronize(); t = time.time()
    if mode == "Légende":
        out = clean_caption(vlm_generate(model, proc, image, CAPTION_PROMPT))
    else:
        cap = user_caption.strip() or clean_caption(vlm_generate(model, proc, image, CAPTION_PROMPT))
        raw = vlm_generate(model, proc, image, JUDGE_PROMPT.format(caption=cap), max_new_tokens=5)
        s = parse_score(raw)
        out = f"Légende évaluée : « {cap} »\nScore : {s}/5" if not np.isnan(s) else f"Réponse non exploitable : {raw}"
    torch.cuda.synchronize()
    info = (f"{name} en {prec} — poids : {model.get_memory_footprint() / 1e9:.2f} Go — "
            f"temps : {time.time() - t:.2f} s")
    return out, info

with gr.Blocks(title="Projet 7 — VLMs légers") as demo:
    gr.Markdown("# Projet 7 — VLMs légers pour légender ou juger des images")
    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type="pil", label="Image")
            name_in = gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Modèle")
            prec_in = gr.Radio(PRECISIONS, value="4bit", label="Niveau de quantification")
            mode_in = gr.Radio(["Légende", "Score"], value="Légende", label="Mode")
            cap_in = gr.Textbox(label="Légende à évaluer (mode Score, optionnel)")
            btn = gr.Button("Lancer", variant="primary")
        with gr.Column():
            out_txt = gr.Textbox(label="Résultat", lines=4)
            out_info = gr.Markdown()
    gr.Examples([[str(IMG_DIR / f"{i}.jpg")] for i in IDS[:4]], inputs=[img_in])
    btn.click(demo_fn, [img_in, name_in, prec_in, mode_in, cap_in], [out_txt, out_info])

demo.launch(share=True, debug=False)

## Mettre à jour GitHub
1. *Fichier → Enregistrer une copie dans GitHub* : dépôt `vlm-compacts`, branche **ta branche** (wafae, meriem ou fatima), chemin **`notebooks/03_juge_demo.ipynb`**, avec un message qui dit ce qui a été fait, par exemple `Ajout des graphiques de comparaison`.
2. Exécute la cellule ci-dessous, décompresse le zip et ajoute les fichiers dans le dossier `results/` de ta branche, puis ouvre une pull request vers `main` (*Add file → Upload files*).

In [ ]:
# Rassemble les résultats légers (CSV, JSON, PNG) à pousser — jamais les images ni les modèles
import zipfile, glob
zip_path = ROOT / "results_fatima.zip"
files = []
for pat in ['notes_humaines_*.csv', 'judge_*.csv', 'correlations_juge.csv', 'correlation_juge.png']:
    files += glob.glob(str(RES_DIR / pat))
with zipfile.ZipFile(zip_path, "w") as z:
    for f in sorted(set(files)):
        z.write(f, arcname=Path(f).name)
print(len(set(files)), "fichiers →", zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception:
    pass